# Step 4 — Final Analysis: Error Breakdown & Qualitative Study

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score

# Set paths
RESULTS_DIR = '../results'
LLM_PREDS_PATH = os.path.join(RESULTS_DIR, 'llm_predictions.csv')
DAVIDSON_PREDS_PATH = os.path.join(RESULTS_DIR, 'davidson_predictions.csv')

# Load data
df_llm = pd.read_csv(LLM_PREDS_PATH)
df_baseline = pd.read_csv(DAVIDSON_PREDS_PATH)

print(f"Loaded {len(df_llm)} LLM predictions and {len(df_baseline)} baseline predictions.")

## 1. Confusion Matrices

We compare the binary predictions (Toxic vs. Non-Toxic) for the baseline classifier and the three LLM strategies.

In [ ]:
def plot_cm(y_true, y_pred, title, ax):
    # Filter out unparseable LLM results (-1)
    mask = y_pred != -1
    cm = confusion_matrix(y_true[mask], y_pred[mask])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Toxic', 'Toxic'])
    disp.plot(cmap='Blues', ax=ax, values_format='d')
    ax.set_title(title)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Baseline (on the subset used for LLM evaluation to ensure comparability)
# We'll use the LLM dataframe as it contains the subset of Davidson tweets tested against the LLM
plot_cm(df_llm['is_toxic'], df_llm['prediction'], "Baseline (Jigsaw Classifier)", axes[0, 0])

# LLM Strategies
plot_cm(df_llm['is_toxic'], df_llm['zero_shot_pred'], "LLM Zero-Shot", axes[0, 1])
plot_cm(df_llm['is_toxic'], df_llm['few_shot_pred'], "LLM Few-Shot", axes[1, 0])
plot_cm(df_llm['is_toxic'], df_llm['cot_pred'], "LLM Chain-of-Thought", axes[1, 1])

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'final_confusion_matrices.png'))
plt.show()

## 2. Detailed Error Breakdown

We use the LLM's multiclass labels (Hate Speech, Offensive, Neither) as a proxy to understand where each method fails.

In [ ]:
# Normalize raw LLM labels for categorization
def categorize_raw(raw_text):
    text = str(raw_text).lower()
    if 'hate' in text: return 'Hate Speech'
    if 'offensive' in text: return 'Offensive'
    if 'neither' in text: return 'Neutral'
    return 'Unparseable'

df_llm['category'] = df_llm['cot_raw'].apply(categorize_raw)

methods = {
    'Baseline': 'prediction',
    'Zero-Shot': 'zero_shot_pred',
    'Few-Shot': 'few_shot_pred',
    'Chain-of-Thought': 'cot_pred'
}

error_stats = []

for name, col in methods.items():
    # Filter for valid predictions
    valid_df = df_llm[df_llm[col] != -1].copy()
    errors = valid_df[valid_df[col] != valid_df['is_toxic']]
    
    breakdown = errors.groupby('category').size()
    for cat, count in breakdown.items():
        if cat != 'Unparseable':
            error_stats.append({'Method': name, 'Category': cat, 'Error Count': count})

df_errors = pd.DataFrame(error_stats)
summary_pivot = df_errors.pivot(index='Method', columns='Category', values='Error Count').fillna(0).astype(int)

print("Error Breakdown by Method and Category:")
print(summary_pivot)

plt.figure(figsize=(12, 6))
sns.barplot(x='Method', y='Error Count', hue='Category', data=df_errors, palette='muted')
plt.title('Error Distribution by Method and Content Category (LLM Proxy)')
plt.ylabel('Number of Errors')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(RESULTS_DIR, 'detailed_error_breakdown_all_methods.png'))
plt.show()

## 3. Qualitative Disagreement Analysis

We examine cases where the Jigsaw-trained classifier and the LLM (CoT) disagree.

In [ ]:
# Filter for disagreements between Baseline and CoT LLM
disagreements = df_llm[(df_llm['prediction'] != df_llm['cot_pred']) & (df_llm['cot_pred'] != -1)]

print(f"Found {len(disagreements)} disagreement cases.")

# Sample some interesting cases
print("\n--- Sample Disagreement Cases ---")
samples = disagreements.sample(min(5, len(disagreements)), random_state=42)
for i, row in samples.iterrows():
    print(f"Tweet: {row['text']}")
    print(f"Ground Truth (is_toxic): {row['is_toxic']}")
    print(f"Baseline Pred: {row['prediction']}")
    print(f"LLM Pred: {row['cot_pred']} ({row['category']})")
    print(f"LLM Reasoning: {row['cot_raw']}")
    print("-" * 30)

## 4. Prompt Variation & Reliability

We analyze how the prompting strategy affects the F1 score and the model's tendency to parse correctly.

In [ ]:
strategies = ['zero_shot_pred', 'few_shot_pred', 'cot_pred']
names = ['Zero-Shot', 'Few-Shot', 'Chain-of-Thought']

metrics = []
for strat, name in zip(strategies, names):
    mask = df_llm[strat] != -1
    f1 = f1_score(df_llm['is_toxic'][mask], df_llm[strat][mask], average='macro')
    parse_rate = mask.mean() * 100
    metrics.append({'Strategy': name, 'Macro F1': f1, 'Parse Rate (%)': parse_rate})

df_metrics = pd.DataFrame(metrics)
print(df_metrics)

df_metrics.plot(x='Strategy', y='Macro F1', kind='bar', legend=False, color='teal')
plt.title('LLM Macro F1 Score by Prompt Strategy')
plt.ylim(0, 1)
plt.ylabel('Macro F1')
plt.savefig(os.path.join(RESULTS_DIR, 'prompt_variation_comparison.png'))
plt.show()